## 0. Configurando sessão spark

In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

In [10]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [11]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [12]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_alfabetizacao_municipio = f"{par_source_project}.silver.meta_alfabetizacao_municipio"

par_source_gold_tempo = f"{par_source_project}.gold.dim_tempo"

## 3. Transformações

### 3.1. Anos com meta

In [13]:
dim_tempo = (
    spark.range(2023, 2031).withColumnRenamed("id","ano")
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("tipo_ano",
        F.when(F.col("ano") <= 2024, F.lit("realizado")).otherwise(F.lit("projetado")))
)

## 5. Armazenamento no BQ

In [ ]:
(
    dim_tempo.write.format("bigquery")
    .option("table", par_source_gold_tempo)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)